In [2]:
suppressPackageStartupMessages({
    library(Seurat)
    library(Signac)
    library(Matrix)
    library(dplyr)
    library(SummarizedExperiment)
    library(ggplot2)
    library(tibble)
    library(rstatix)
    library(reshape2)
    library(tidyr)
    library(scales)
    '%ni%' <- Negate('%in%')
    source("2_HelperFunction.R")
    source("VAF_dist_function.R")
    source("../../global/quantifyMMB.R")
})    

Warning message in ifelse(SIFT == "-", NA, as.numeric(sub(".*\\(([^)]+)\\)", "\\1", :
“NAs introduced by coercion”
Warning message in ifelse(PolyPhen == "-", NA, as.numeric(sub(".*\\(([^)]+)\\)", :
“NAs introduced by coercion”


In [3]:
cols <- c(
  "CTRL_Glu"    = "#BBD6FF",
  "CTRL_Gal_1D" = "#2780FF",
  "CTRL_Gal_3D" = "#0040A8",
  "KI36_Glu"    = "#F0BBFF",
  "KI36_Gal_1D" = "#960096",
  "KI36_Gal_3D" = "#5A0050",
  "Doublet"     = "black", 
  "Negative"    = "grey")

In [4]:
# Load data
Gal <- readRDS("../output/1_preprocessed_HEK_POLG_galactose_seurat.rds")

mmat_CTRL <- readRDS("../output/1_mmat_CTRL_Gal.rds")
mmat_KI36 <- readRDS("../output/1_mmat_KI36_Gal.rds")

cov_CTRL <- readRDS("../output/1_scCoverage.CTRL_Gal.rds")
cov_KI36 <- readRDS("../output/1_scCoverage.KI36_Gal.rds")

In [5]:
# extract barcode
CTRL_GluD3_CB <- WhichCells(Gal, idents = "CTRL_Glu"    )
CTRL_GalD1_CB <- WhichCells(Gal, idents = "CTRL_Gal_1D" )
CTRL_GalD3_CB <- WhichCells(Gal, idents = "CTRL_Gal_3D" )
KI36_GluD3_CB <- WhichCells(Gal, idents = "KI36_Glu"    )
KI36_GalD1_CB <- WhichCells(Gal, idents = "KI36_Gal_1D" )
KI36_GalD3_CB <- WhichCells(Gal, idents = "KI36_Gal_3D" )

CTRL_CB = c(CTRL_GluD3_CB, CTRL_GalD1_CB, CTRL_GalD3_CB)
KI36_CB = c(KI36_GluD3_CB, KI36_GalD1_CB, KI36_GalD3_CB)

In [6]:
mmat_het_CTRL <- mgatk_filter(mmat_CTRL)
mmat_het_KI36 <- mgatk_filter(mmat_KI36)

In [7]:
meta_het_CTRL <- prepare_mtVAR_meta(mmat_het_CTRL, mito.annotation)

matrix_het_CTRL <- extract_variant_matrices(meta_het_CTRL, assay(mmat_het_CTRL))

suppressWarnings({ # reshape2 is deprecated
    var_het_CTRL_disease    <- top_vaf_distribution(matrix_het_CTRL$disease)
    var_het_CTRL_truncating <- top_vaf_distribution(matrix_het_CTRL$truncating)
    var_het_CTRL_missense   <- top_vaf_distribution(matrix_het_CTRL$missense)
    var_het_CTRL_synonymous <- top_vaf_distribution(matrix_het_CTRL$synonymous)
})


In [8]:
var_het_CTRL_disease$group    = paste0(var_het_CTRL_disease$variant,    "_" ,  var_het_CTRL_disease$hash.ID   )   
var_het_CTRL_truncating$group = paste0(var_het_CTRL_truncating$variant, "_" ,  var_het_CTRL_truncating$hash.ID)
var_het_CTRL_missense$group   = paste0(var_het_CTRL_missense$variant,   "_" ,  var_het_CTRL_missense$hash.ID  )  
var_het_CTRL_synonymous$group = paste0(var_het_CTRL_synonymous$variant, "_" ,  var_het_CTRL_synonymous$hash.ID)
  

In [9]:
options(repr.plot.width = 3, repr.plot.height = 3, repr.plot.res = 300)

vars <- list(
    var_het_CTRL_disease,   
    var_het_CTRL_truncating,
    var_het_CTRL_missense,  
    var_het_CTRL_synonymous
)

titles <- c(
  "Disease variants in CTRL",
  "Truncating variants in CTRL",
  "Missense variants in CTRL",
  "Synonymous variants in CTRL"
)

file_tags <- c("Disease", "Truncating", "Missense", "Synonymous")

plots <- lapply(seq_along(vars), function(i) {

  df <- vars[[i]] %>%
    group_by(variant, hash.ID) %>%
    summarise(mean = mean(VAF), .groups = "drop") %>%
    filter(hash.ID != "CTRL_Gal_1D") %>%
    pivot_wider(names_from = "hash.ID", values_from = "mean")

  # get common axis limits for x and y
  # ims <- range(c(df$CTRL_Gal_3D, df$CTRL_Glu), na.rm = TRUE)

  p <- ggplot(df, aes(x = CTRL_Gal_3D, y = CTRL_Glu)) +
    geom_point() +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    coord_equal(xlim = c(0,0.5), ylim = c(0,0.5), expand = TRUE) +
    theme_classic() +
    xlab("Mean VAF Galactose 3 Days") +
    ylab("Mean VAF Glucose") +
    ggtitle(titles[i])

  # save plot
  ggsave(
    filename = paste0("../plot/Ext_Fig9e_2_1_CTRL_", file_tags[i], "_meanVAF_GalvsGlu.pdf"),
    plot = p,
    device = "pdf",
    width = 3,
    height = 3
  )

  p
})

In [10]:
var_het_CTRL_truncating %>% 
    group_by(variant, hash.ID) %>% 
    summarise(mean = mean(VAF)) %>%  ungroup()%>% 
    group_by(variant) %>% 
    mutate(delta = mean - mean[hash.ID == "CTRL_Glu"]) %>% ungroup()%>% 
    arrange(desc(abs(delta))) %>% filter(hash.ID != "CTRL_Glu")%>% 
    group_by(hash.ID) %>% slice_head(n = 3)

`summarise()` has grouped output by 'variant'. You can override using the
`.groups` argument.


variant,hash.ID,mean,delta
<fct>,<fct>,<dbl>,<dbl>
13414G>A,CTRL_Gal_1D,0.019591253,0.0023685793
15710C>T,CTRL_Gal_1D,0.023956171,0.0017981099
5920G>A,CTRL_Gal_1D,0.002816467,-0.0007851472
15710C>T,CTRL_Gal_3D,0.034812703,0.0126546412
13414G>A,CTRL_Gal_3D,0.028877303,0.0116546294
9376G>A,CTRL_Gal_3D,0.011440624,0.0038474709


In [11]:
var_het_CTRL_missense %>% 
    group_by(variant, hash.ID) %>% 
    summarise(mean = mean(VAF)) %>%  ungroup()%>% 
    group_by(variant) %>% 
    mutate(delta = mean - mean[hash.ID == "CTRL_Glu"]) %>% ungroup()%>% 
    arrange(desc(abs(delta))) %>% filter(hash.ID != "CTRL_Glu")%>% 
    group_by(hash.ID) %>% slice_head(n = 3)

`summarise()` has grouped output by 'variant'. You can override using the
`.groups` argument.


variant,hash.ID,mean,delta
<fct>,<fct>,<dbl>,<dbl>
10762G>A,CTRL_Gal_1D,0.02641686,0.003388113
8984C>T,CTRL_Gal_1D,0.04581763,0.003317588
15596G>A,CTRL_Gal_1D,0.05229242,0.003123950
15596G>A,CTRL_Gal_3D,0.07396066,0.024792185
8984C>T,CTRL_Gal_3D,0.06289956,0.020399526
11672A>T,CTRL_Gal_3D,0.06139300,0.017692269


In [12]:
var_het_CTRL_synonymous %>% 
    group_by(variant, hash.ID) %>% 
    summarise(mean = mean(VAF)) %>%  ungroup()%>% 
    group_by(variant) %>% 
    mutate(delta = mean - mean[hash.ID == "CTRL_Glu"]) %>% ungroup()%>% 
    arrange(desc(abs(delta))) %>% filter(hash.ID != "CTRL_Glu")%>% 
    group_by(hash.ID) %>% slice_head(n = 3)

`summarise()` has grouped output by 'variant'. You can override using the
`.groups` argument.


variant,hash.ID,mean,delta
<fct>,<fct>,<dbl>,<dbl>
7211G>A,CTRL_Gal_1D,0.25128579,0.010637616
7931C>T,CTRL_Gal_1D,0.02863556,0.003422079
10762G>A,CTRL_Gal_1D,0.02641686,0.003388113
13617T>C,CTRL_Gal_3D,0.92884267,-0.024103568
3970C>T,CTRL_Gal_3D,0.06257495,0.019852310
7610C>T,CTRL_Gal_3D,0.04396568,0.013677517


In [13]:
var_het_CTRL_disease %>% 
    group_by(variant, hash.ID) %>% 
    summarise(mean = mean(VAF)) %>%  ungroup()%>% 
    group_by(variant) %>% 
    mutate(delta = mean - mean[hash.ID == "CTRL_Glu"]) %>% ungroup()%>% 
    arrange(desc(abs(delta))) %>% filter(hash.ID != "CTRL_Glu")%>% 
    group_by(hash.ID) %>% slice_head(n = 3)

`summarise()` has grouped output by 'variant'. You can override using the
`.groups` argument.


variant,hash.ID,mean,delta
<fct>,<fct>,<dbl>,<dbl>
8313G>A,CTRL_Gal_1D,0.0260063864,0.0018305516
10197G>A,CTRL_Gal_1D,0.0051365007,0.0003508335
4450G>A,CTRL_Gal_1D,0.0002042154,-0.0001946020
8313G>A,CTRL_Gal_3D,0.0372580982,0.0130822634
10197G>A,CTRL_Gal_3D,0.0071748596,0.0023891925
14849T>C,CTRL_Gal_3D,0.0008169898,0.0004669816


In [14]:
meta_het_KI36 <- prepare_mtVAR_meta(mmat_het_KI36, mito.annotation)

matrix_het_KI36 <- extract_variant_matrices(meta_het_KI36, assay(mmat_het_KI36))

suppressWarnings({ # reshape2 is deprecated
    var_het_KI36_disease    <- top_vaf_distribution(matrix_het_KI36$disease)
    var_het_KI36_truncating <- top_vaf_distribution(matrix_het_KI36$truncating)
    var_het_KI36_missense   <- top_vaf_distribution(matrix_het_KI36$missense)
    var_het_KI36_synonymous <- top_vaf_distribution(matrix_het_KI36$synonymous)
})

var_het_KI36_synonymous$group = paste0(var_het_KI36_synonymous$variant, "_" ,var_het_KI36_synonymous$hash.ID)
var_het_KI36_disease$group = paste0(var_het_KI36_disease$variant, "_" ,var_het_KI36_disease$hash.ID)
var_het_KI36_missense$group = paste0(var_het_KI36_missense$variant, "_" ,var_het_KI36_missense$hash.ID)
var_het_KI36_truncating$group = paste0(var_het_KI36_truncating$variant, "_" ,var_het_KI36_truncating$hash.ID)


In [15]:
options(repr.plot.width = 3, repr.plot.height = 3, repr.plot.res = 300)

vars <- list(
  var_het_KI36_disease,
  var_het_KI36_truncating,
  var_het_KI36_missense,
  var_het_KI36_synonymous
)

titles <- c(
  "Disease variants in KI36",
  "Truncating variants in KI36",
  "Missense variants in KI36",
  "Synonymous variants in KI36"
)

file_tags <- c("Disease", "Truncating", "Missense", "Synonymous")

plots <- lapply(seq_along(vars), function(i) {
  p <- vars[[i]] %>%
    group_by(variant, hash.ID) %>%
    summarise(mean = mean(VAF), .groups = "drop") %>%
    filter(hash.ID != "KI36_Gal_1D") %>%
    pivot_wider(names_from = "hash.ID", values_from = "mean") %>%
    ggplot(aes(x = KI36_Gal_3D, y = KI36_Glu)) +
    geom_point() +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    coord_equal(xlim = c(0,0.5), ylim = c(0,0.5), expand = TRUE) +
    theme_classic() +
    xlab("Mean VAF Galactose 3 Days") +
    ylab("Mean VAF Glucose") +
    ggtitle(titles[i])

  # save plot
  ggsave(
    filename = paste0("../plot/Fig5e_2_1_KI36_", file_tags[i], "_meanVAF_GalvsGlu.pdf"),
    plot = p,
    device = "pdf",
    width = 3,
    height = 3
  )

  p
})

In [16]:
var_het_KI36_synonymous %>% 
    group_by(variant, hash.ID) %>% 
    summarise(mean = mean(VAF)) %>%  ungroup()%>% 
    group_by(variant) %>% 
    mutate(delta = mean - mean[hash.ID == "KI36_Glu"]) %>% ungroup()%>% 
    arrange(desc(abs(delta))) %>% filter(hash.ID != "KI36_Glu")%>% 
    group_by(hash.ID) %>% slice_head(n = 3)

`summarise()` has grouped output by 'variant'. You can override using the
`.groups` argument.


variant,hash.ID,mean,delta
<fct>,<fct>,<dbl>,<dbl>
10951C>T,KI36_Gal_1D,0.1619080,0.02679167
12741C>T,KI36_Gal_1D,0.1537314,0.02458196
11902G>A,KI36_Gal_1D,0.1556922,0.02430353
6824C>T,KI36_Gal_3D,0.1538238,0.01879406
10951C>T,KI36_Gal_3D,0.1537282,0.01861184
9743A>G,KI36_Gal_3D,0.1536558,0.01799754


In [17]:
var_het_KI36_truncating %>% 
    group_by(variant, hash.ID) %>% 
    summarise(mean = mean(VAF)) %>%  ungroup()%>% 
    group_by(variant) %>% 
    mutate(delta = mean - mean[hash.ID == "KI36_Glu"]) %>% ungroup()%>% 
    arrange(desc(abs(delta))) %>% filter(hash.ID != "KI36_Glu")%>% 
    group_by(hash.ID) %>% slice_head(n = 3)

`summarise()` has grouped output by 'variant'. You can override using the
`.groups` argument.


variant,hash.ID,mean,delta
<fct>,<fct>,<dbl>,<dbl>
13414G>A,KI36_Gal_1D,0.15590228,0.025787951
9376G>A,KI36_Gal_1D,0.06271122,0.008221087
6264G>A,KI36_Gal_1D,0.03642514,-0.004322140
13414G>A,KI36_Gal_3D,0.14932691,0.019212589
9376G>A,KI36_Gal_3D,0.06262031,0.008130173
15797G>A,KI36_Gal_3D,0.04842178,-0.005382646


In [23]:
var_het_KI36_missense %>% 
    group_by(variant, hash.ID) %>% 
    summarise(mean = mean(VAF)) %>%  ungroup()%>% 
    group_by(variant) %>% 
    mutate(delta = mean - mean[hash.ID == "KI36_Glu"]) %>% ungroup()%>% 
    arrange(desc(abs(delta))) %>% filter(hash.ID != "KI36_Glu")%>% 
    group_by(hash.ID) %>% slice_head(n = 3)

`summarise()` has grouped output by 'variant'. You can override using the
`.groups` argument.


variant,hash.ID,mean,delta
<fct>,<fct>,<dbl>,<dbl>
8791C>T,KI36_Gal_1D,0.1631151,0.02624976
13766C>T,KI36_Gal_1D,0.1621613,0.02550295
13330C>T,KI36_Gal_1D,0.1674560,0.02532037
8791C>T,KI36_Gal_3D,0.1563218,0.01945649
11178C>T,KI36_Gal_3D,0.1532066,0.01872282
13330C>T,KI36_Gal_3D,0.1608392,0.01870359


In [18]:
var_het_KI36_disease %>% 
    group_by(variant, hash.ID) %>% 
    summarise(mean = mean(VAF)) %>%  ungroup()%>% 
    group_by(variant) %>% 
    mutate(delta = mean - mean[hash.ID == "KI36_Glu"]) %>% ungroup()%>% 
    arrange(desc(abs(delta))) %>% filter(hash.ID != "KI36_Glu")%>% 
    group_by(hash.ID) %>% slice_head(n = 3)

`summarise()` has grouped output by 'variant'. You can override using the
`.groups` argument.


variant,hash.ID,mean,delta
<fct>,<fct>,<dbl>,<dbl>
8313G>A,KI36_Gal_1D,0.22802165,0.008516601
10197G>A,KI36_Gal_1D,0.03715980,0.004041284
14568C>T,KI36_Gal_1D,0.01657411,-0.002374556
8313G>A,KI36_Gal_3D,0.22783795,0.008332900
10197G>A,KI36_Gal_3D,0.03801271,0.004894202
14568C>T,KI36_Gal_3D,0.01680580,-0.002142871


In [21]:
options(repr.plot.width = 3, repr.plot.height = 3, repr.plot.res = 300)
p_10951_CT <- var_het_KI36_synonymous %>% filter(variant %in% c("10951C>T")) %>% 
    ggplot(aes(x = VAF, group = group, color = hash.ID)) +
    stat_ecdf(geom = "step") +
    scale_color_manual(values = cols) +
    #labs(title = paste("VAF ECDF (mt.10951C>T)"), x = "VAF", y = "Cumulative Probability") +
    theme_classic()  + scale_x_continuous(limits = c(0,1)) +
    labs(title = NULL, x = NULL, y = NULL) +
    theme(legend.position = "none", axis.text.x=element_blank(), axis.text.y=element_blank()) 
 
ggsave(plot = p_10951_CT, "../plot/Fig_5f_10951_CT_ECDF.pdf", height = 3, width = 3, dpi = 300)
#options(repr.plot.width = 3, repr.plot.height = 3, repr.plot.res = 300)
#var_het_KI36_synonymous %>% filter(variant %in% c("10951C>T")) %>% #, "9376G>A", "4214G>A", "6264G>A", "15797G>A")) %>% 
#    ggplot(aes(x = VAF, group = group, color = hash.ID)) +
#    stat_ecdf(geom = "step") +
#    scale_color_manual(values = cols) +
#    labs(x = NULL, y = NULL) +
#    theme_classic() + theme(legend.position = "none") + scale_x_continuous(limits = c(0,0.75), expand = c(0,0)) +
#    scale_y_continuous(limits = c(0.4,1), expand = c(0,0))
 

In [22]:
options(repr.plot.width = 3, repr.plot.height = 3, repr.plot.res = 300)
p_13766CT <- var_het_KI36_missense %>% filter(variant %in% c("13766C>T")) %>% 
    ggplot(aes(x = VAF, group = group, color = hash.ID)) +
    stat_ecdf(geom = "step") +
    scale_color_manual(values = cols) +
    #labs(title = paste("VAF ECDF (mt.13766C>T)"), x = "VAF", y = "Cumulative Probability") +
    theme_classic() + scale_x_continuous(limits = c(0,1)) +
    labs(title = NULL, x = NULL, y = NULL) +
    theme(legend.position = "none", axis.text.x=element_blank(), axis.text.y=element_blank()) 

ggsave(plot = p_13766CT, "../plot/Fig_5f_13766CT_ECDF.pdf", height = 3, width = 3, dpi = 300)

#var_het_KI36_missense %>% filter(variant %in% c("13766C>T")) %>% #, "9376G>A", "4214G>A", "6264G>A", "15797G>A")) %>% 
#    ggplot(aes(x = VAF, group = group, color = hash.ID)) +
#    stat_ecdf(geom = "step") +
#    scale_color_manual(values = cols) +
#    labs(x = NULL, y = NULL) +
#    theme_classic() + theme(legend.position = "none") + scale_x_continuous(limits = c(0,0.75), expand = c(0,0)) +
#    scale_y_continuous(limits = c(0.4,1), expand = c(0,0))
 

In [23]:
options(repr.plot.width = 3, repr.plot.height = 3, repr.plot.res = 300)
p_13414GA <- var_het_KI36_truncating %>% filter(variant %in% c("13414G>A")) %>% 
    ggplot(aes(x = VAF, group = group, color = hash.ID)) +
    stat_ecdf(geom = "step") +
    scale_color_manual(values = cols) +
    #labs(title = paste("VAF ECDF (mt.13414G>A)"), x = "VAF", y = "Cumulative Probability") +
    theme_classic()  + scale_x_continuous(limits = c(0,1))+
    labs(title = NULL, x = NULL, y = NULL) +
    theme(legend.position = "none", axis.text.x=element_blank(), axis.text.y=element_blank()) 


ggsave(plot = p_13414GA, "../plot/Fig_5f_13414GA_ECDF.pdf", height = 3, width = 3, dpi = 300)

#var_het_KI36_truncating %>% filter(variant %in% c("13414G>A")) %>% 
#    ggplot(aes(x = VAF, group = group, color = hash.ID)) +
#    stat_ecdf(geom = "step") +
#    scale_color_manual(values = cols) +
#    labs(x = NULL, y = NULL) +
#    theme_classic() + theme(legend.position = "none") + scale_x_continuous(limits = c(0,0.75), expand = c(0,0)) +
#    scale_y_continuous(limits = c(0.4,1), expand = c(0,0))

In [25]:
options(repr.plot.width = 3, repr.plot.height = 3, repr.plot.res = 300)
p_8313GA <- var_het_KI36_disease %>% filter(variant %in% c("8313G>A")) %>% 
    ggplot(aes(x = VAF, group = group, color = hash.ID)) +
    stat_ecdf(geom = "step") +
    scale_color_manual(values = cols) +
    #labs(title = paste("VAF ECDF (mt.8313G>A)"), x = "VAF", y = "Cumulative Probability") +
    theme_classic() + scale_x_continuous(limits = c(0,1)) +
    labs(title = NULL, x = NULL, y = NULL) +
    theme(legend.position = "none", axis.text.x=element_blank(), axis.text.y=element_blank()) 

ggsave(plot = p_8313GA, "../plot/Fig_5f_8313GA_ECDF.pdf", height = 3, width = 3, dpi = 300)
